# 实验九 · Transformer - 翻译任务

在上次实验中，我们在图结构上，使用 Attention 机制进行训练和预测，初步感受到了所谓“注意力机制”的注意力到底体现在哪里。但由于大家平时对图结构接触较少，对注意力机制的感受可能仍然不够深刻。这次实验，我们将回归 Attention 最原始的应用——语言的处理。这也是现代大语言模型最为基础、关键的技术。

这次实验的核心目标是让同学们构建简单的 Transformer，用来处理简单的**英译中**任务。在 Transformer 之前，人们还尝试过很多模型，用于处理文本，比如经典的 RNN、GRU、LSTM。在实验末尾，我们会给出代码段，展示这些模型各自的特点。

> 得一提的是，《Attention is All You Need》这篇论文是 2017 发表的，至今马上也年满十年。很难想象在这个技术迭代速度下，主流模型又会采用什么技术和架构呢……

实验中使用到的数据集为 [英中短句.tsv](英中短句.tsv)。之前我们用的是 `.csv` 文件，二者本质都是表格，唯一的区别在于 `tsv` 文件是用空格来分割的。


In [ ]:
import csv
import math
import os
import re
import time
import unicodedata
from collections import Counter
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

SEED = 2026
torch.manual_seed(SEED)
torch.set_num_threads(min(4, os.cpu_count() or 1))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_FILE = Path("英中短句.tsv")
CHECKPOINT_FILE = Path("参考权重.pt")
TEST_CASES = [
    ("否定", "I agree.", "我同意。"),
    ("否定", "I don't agree.", "我不同意。"),
    ("数量", "I have two books.", "我有两本书。"),
    ("数量", "I have three books.", "我有三本书。"),
    ("主客体", "Tom called Mary.", "汤姆给玛丽打了电话。"),
    ("主客体", "Mary called Tom.", "玛丽给汤姆打了电话。"),
]
DECODE_EXAMPLE = ("额外对照", "Good morning.", "早上好。")
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<unk>"]
PAD_ID, BOS_ID, EOS_ID, UNK_ID = range(len(SPECIAL_TOKENS))
MODEL_CONFIG = {
    "d_model": 96,
    "nhead": 4,
    "layers": 2,
    "feedforward": 192,
    "dropout": 0.1,
}

print(f"PyTorch {torch.__version__} | {DEVICE} | CPU 线程数 {torch.get_num_threads()}")

PyTorch 2.14.1 | cpu | CPU 线程数 4


## Q1 · 一个词元怎样成为一个向量？

### 任务 1 · 预测切分结果

回忆课上的内容。我们并没有办法直接对一段文本进行计算，计算机只认数字而不是文本。因此，我们需要有一种方法，能够将一段文本映射成可用于计算的数据类型。Token 这个词本意有令牌，其实也就可以理解成，让文本手持一个令牌，进入计算流程中参与计算。得到的计算结果我们再利用这个令牌来对应原来的文本。随着技术的普及，结合处理方式本身，大家也逐渐翻译为**词元**。

正如上文所述，Token 是分词器交给模型处理的一个序列单位。这里的切分规则并不固定，切分规则决定它可能是一个词、一个字、一个子词或一个标点。比如「同意」可以作为一个词元，也可以切成「同」和「意」。用子词切分时，英文 `playing` 也可能被切成 `play` 和 `ing`。

> 在实际的应用中，存在一个“停用词”词表，模型在处理的时候会专门跳过这些词语，保证这些词语语义的完整性。比如加入 `deepseek` 这个专用词，模型在处理的时候就不会将其拆分为 `deep`,`seek`，保证一些专用词不被破坏。

本实验将英文转成小写，再按单词、数字串和标点切分，`don't` 这样的缩写保留为一个词元。中文按非空白字符切分，句号也占一个位置。

> 这里直接给出了分词的代码，不需要同学们查询相应词库的用法。但在运行前，请思考：_按照分词规则，下面的句子会被拆分成什么内容_。
>
> 此外，这里还涉及到 python 中字符串的正则匹配表达式，具体用法为 `r{...}`。匹配的代码已经给出，感兴趣的同学自行查阅相关用法 ~~(其实是助教也不怎么会用呜呜呜呜)~~


In [2]:
def tokenize_en(text):
    text = unicodedata.normalize("NFKC", text).lower().replace("’", "'")
    return re.findall(r"[a-z]+(?:'[a-z]+)?|\d+|[^\w\s]", text)


def tokenize_zh(text):
    return [char for char in unicodedata.normalize("NFKC", text) if not char.isspace()]

In [3]:
en_text = "I don’t agree!"
zh_text = "我不同意。"

for text, tokenize in [(en_text, tokenize_en), (zh_text, tokenize_zh)]:
    tokens = tokenize(text)
    print(f"原文     {text}")
    print(f"空白切分 {text.split()}")
    print(f"课堂词元 {tokens}，共 {len(tokens)} 个\n")

原文     I don’t agree!
空白切分 ['I', 'don’t', 'agree!']
课堂词元 ['i', "don't", 'agree', '!']，共 4 个

原文     我不同意。
空白切分 ['我不同意。']
课堂词元 ['我', '不', '同', '意', '。']，共 5 个



### 任务 2 · 查词元 ID，观察特殊词元

**词元 ID** 是词元在词表里的整数下标。源语言和目标语言各有一张词表。这里词表的存储可以直接理解为 python/C 中的**数组**，即 `words[]`，这样我们直接对下标进行操作即可。注意，这里的 ID 是对词表的位置选择，和 token 本身无关，因此同一个整数在两张表里可以指向不同的词元。

这里，我们从训练集建立词表，普通词元至少出现两次才收录（这是为了过滤掉低频噪声）。查表时，未收录的词元映射到 `<unk>`。`<bos>` 和 `<eos>` 标记序列的开始和结束，`<pad>` 用于把不同长度的序列补成同样长。

> 这里 `<XXX>` 的用法称作**正则**，和前面提到的 `r{...}` 用法基本同理，都是去匹配文本段落中的特定内容进而进行处理。缩写含义为：`<bos>` = `beginning of sentence`, `<eos>` = `end of sentence`, `<pad>` = `padding`。方便理解和记忆。


In [4]:
def build_vocab(sentences, tokenize, min_freq=2):
    counts = Counter(token for sentence in sentences for token in tokenize(sentence))
    tokens = sorted(token for token, count in counts.items() if count >= min_freq)
    return SPECIAL_TOKENS + tokens


if not DATA_FILE.is_file():
    raise FileNotFoundError(
        "请将「英中短句.tsv」与 Notebook 放在同一目录，并从该目录启动 Notebook。"
    )

splits = {"train": [], "val": []}
with DATA_FILE.open(encoding="utf-8", newline="") as data_file:
    for row in csv.DictReader(data_file, delimiter="\t"):
        splits[row["split"]].append((row["en"], row["zh"]))

train_pairs, val_pairs = splits["train"], splits["val"]
src_vocab = build_vocab((en for en, _ in train_pairs), tokenize_en)
tgt_vocab = build_vocab((zh for _, zh in train_pairs), tokenize_zh)
src_to_id = {token: i for i, token in enumerate(src_vocab)}
tgt_to_id = {token: i for i, token in enumerate(tgt_vocab)}


def source_key(sentence):
    tokens = tokenize_en(sentence)
    while tokens and not any(char.isalnum() for char in tokens[-1]):
        tokens.pop()
    return tuple(tokens)


train_keys = {source_key(en) for en, _ in train_pairs}
val_keys = {source_key(en) for en, _ in val_pairs}
held_out = {source_key(DEMO_SENTENCE)} | {
    source_key(en) for _, en, _ in [*TEST_CASES, DECODE_EXAMPLE]
}
assert train_keys.isdisjoint(val_keys), "同一个英文原句出现在训练集和验证集"
assert held_out.isdisjoint(train_keys | val_keys), (
    "有演示句或测试句出现在训练集或验证集"
)

print(f"训练 {len(train_pairs):,} 对 | 验证 {len(val_pairs):,} 对")
print(f"英文词表 {len(src_vocab):,} | 中文词表 {len(tgt_vocab):,}")

训练 21,850 对 | 验证 1,150 对
英文词表 3,793 | 中文词表 2,765


In [5]:
def ids_for(tokens, vocab):
    # TODO 1 · 依次查每个词元的 ID。缺失词元使用 UNK_ID。
    # 返回 list[int]，保留原顺序和重复词元。
    raise NotImplementedError

In [6]:
check_vocab = {"hello": 8, "world": 9, "<pad>": PAD_ID}
assert ids_for(["world", "missing", "hello", "world", "<pad>"], check_vocab) == [
    9,
    UNK_ID,
    8,
    9,
    PAD_ID,
]
assert ids_for([], check_vocab) == []
print("词元查表通过。保留了顺序、重复词元和有效的 ID 0，并处理了未登录词。")

NotImplementedError: 

下面是一个样例，用于判断同学们上面完成的函数是否正确。同学们观察前后两个词表的不同之处，处理是否正确。


In [ ]:
id_example = "I agree, I agree. zqxvtoken!"
id_tokens = ["<bos>", *tokenize_en(id_example), "<eos>"]
example_ids = ids_for(id_tokens, src_to_id)

for token, index in zip(id_tokens, example_ids):
    print(f"{token:>12} → {index:>4} → {src_vocab[index]}")

shared_id = src_to_id["i"]
print(f"同一个 ID {shared_id} 在两张词表中对应")
print(f"英文 {src_vocab[shared_id]!r}，中文 {tgt_vocab[shared_id]!r}")

### 任务 3 · 用 ID 查词嵌入，比较两个位置

**词嵌入（embedding）** 把一个 ID 变成浮点向量。若词表有 $V$ 个词元，嵌入维度为 $d$，模型就有一张 $V \times d$ 的可训练表 $E$。这里我们引入 `torch` 中的 `nn.Embedding`，直接完成嵌入的工作。当 token ID 是 $i$ 时，`nn.Embedding` 取出第 $i$ 行 $E[i]$。本实验中 $d=96$，英文和中文各用自己的嵌入表。

下面载入本实验的翻译模型和参考权重。同学们注意观察两件事：

1. 预测同一句里两个 `i` 查出的向量是否一样，再比较加入位置编码后的向量。
2. 试着交换 `i` 和 `agree`，同时保留两个 `i`，观察哪些位置发生了变化。


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_positions=256):
        super().__init__()
        positions = torch.arange(max_positions, dtype=torch.float32)[:, None]
        frequencies = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )
        table = torch.zeros(max_positions, d_model)
        table[:, 0::2] = torch.sin(positions * frequencies)
        table[:, 1::2] = torch.cos(positions * frequencies)
        self.register_buffer("table", table, persistent=False)

    def forward(self, embeddings):
        return embeddings + self.table[: embeddings.size(1)].to(dtype=embeddings.dtype)


class TranslationTransformer(nn.Module):
    def __init__(
        self, src_size, tgt_size, d_model, nhead, layers, feedforward, dropout
    ):
        super().__init__()
        self.scale = math.sqrt(d_model)
        self.src_embedding = nn.Embedding(src_size, d_model, padding_idx=PAD_ID)
        self.tgt_embedding = nn.Embedding(tgt_size, d_model, padding_idx=PAD_ID)
        self.positions = PositionalEncoding(d_model)
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=layers,
            num_decoder_layers=layers,
            dim_feedforward=feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.to_vocab = nn.Linear(d_model, tgt_size)

    def encode(self, src):
        src_pad = src.eq(PAD_ID)
        src_emb = self.positions(self.src_embedding(src) * self.scale)
        memory = self.transformer.encoder(src_emb, src_key_padding_mask=src_pad)
        return memory, src_pad

    def decode(self, tgt_in, memory, src_pad, tgt_mask):
        tgt_emb = self.positions(self.tgt_embedding(tgt_in) * self.scale)
        hidden = self.transformer.decoder(
            tgt_emb,
            memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_in.eq(PAD_ID),
            memory_key_padding_mask=src_pad,
        )
        return self.to_vocab(hidden)

    def forward(self, src, tgt_in, tgt_mask):
        memory, src_pad = self.encode(src)
        return self.decode(tgt_in, memory, src_pad, tgt_mask)


def new_model():
    return TranslationTransformer(len(src_vocab), len(tgt_vocab), **MODEL_CONFIG).to(
        DEVICE
    )


def library_causal_mask(length, device):
    additive_mask = nn.Transformer.generate_square_subsequent_mask(
        length, device=device
    )
    return additive_mask.isneginf()

In [ ]:
if not CHECKPOINT_FILE.is_file():
    raise FileNotFoundError("请把「参考权重.pt」放到 Notebook 所在目录。")

checkpoint = torch.load(CHECKPOINT_FILE, map_location="cpu", weights_only=True)
if (
    checkpoint["config"] != MODEL_CONFIG
    or checkpoint["src_vocab"] != src_vocab
    or checkpoint["tgt_vocab"] != tgt_vocab
):
    raise ValueError("参考权重、模型配置和词表应来自同一份实验材料。")

reference_model = new_model()
reference_model.load_state_dict(checkpoint["state_dict"])
reference_model.eval()
print(f"参考权重 | 第 {checkpoint['epoch']} 轮 | 验证损失 {checkpoint['val_loss']:.3f}")

In [ ]:
embedding_example = "I agree, I agree."
repeated_token = "i"
embedding_tokens = ["<bos>", *tokenize_en(embedding_example), "<eos>"]
embedding_ids = torch.tensor([ids_for(embedding_tokens, src_to_id)], device=DEVICE)

with torch.no_grad():
    embeddings = reference_model.src_embedding(embedding_ids)
    with_positions = reference_model.positions(embeddings * reference_model.scale)

print(f"嵌入表形状  {tuple(reference_model.src_embedding.weight.shape)}")
print(f"ID 张量形状 {tuple(embedding_ids.shape)}")
print(f"嵌入后形状  {tuple(embeddings.shape)}")

matches = [i for i, token in enumerate(embedding_tokens) if token == repeated_token]
if len(matches) >= 2:
    first, second = matches[:2]
    print(f"{repeated_token!r} 出现在位置 {first} 和 {second}（从 0 开始计数）")
    print(f"两个 ID  {embedding_ids[0, [first, second]].tolist()}")
    print("查表所得的前 4 个分量", embeddings[0, [first, second], :4].tolist())
    print("查表向量相同", torch.equal(embeddings[0, first], embeddings[0, second]))
    print(
        "缩放并加位置编码后相同",
        torch.equal(with_positions[0, first], with_positions[0, second]),
    )
else:
    print(f"请在例句中至少保留两个 {repeated_token!r}。")

### 任务 4 · 固定词元和位置，只改变上下文

下面两句的 `i` 都在 `<bos>` 后的第一个位置。先预测这两个位置的查表向量、加入位置编码后的向量和编码器输出是否相同，再运行代码。然后只改第二句的标点再试一次。记录哪一步开始受到句中其他词元的影响。


In [ ]:
context_sentences = ["I agree.", "I don't agree."]
context_tokens = [tokenize_en(sentence) for sentence in context_sentences]
assert all(tokens and tokens[0] == "i" for tokens in context_tokens), (
    "请让两句都以 I 开头。"
)

context_rows = [
    torch.tensor([BOS_ID, *ids_for(tokens, src_to_id), EOS_ID], device=DEVICE)
    for tokens in context_tokens
]
context_ids = pad_sequence(context_rows, batch_first=True, padding_value=PAD_ID)

with torch.no_grad():
    context_embeddings = reference_model.src_embedding(context_ids)
    encoder_inputs = reference_model.positions(
        context_embeddings * reference_model.scale
    )
    context_vectors, _ = reference_model.encode(context_ids)

print(
    "词元 i 的查表向量相同",
    torch.equal(context_embeddings[0, 1], context_embeddings[1, 1]),
)
print("加位置编码后相同", torch.equal(encoder_inputs[0, 1], encoder_inputs[1, 1]))
distance = torch.linalg.vector_norm(
    context_vectors[0, 1] - context_vectors[1, 1]
).item()
print(f"两个编码器输出的欧氏距离 {distance:.4f}")

**记录实验结果**

- 改写缩写后，词元数怎样变化？记录两种英文写法的词元列表。
- 两个未登录词查表后发生了什么？它们还能从 ID 反查回原文吗？
- 同一句里重复的 `i` 和不同句子中同一位置的 `i`，分别在哪一步开始得到不同向量？

在这里记录你的观察。


## Q2 · 同一句译文怎样用于训练和生成？

我们提前准备好了 Token ID 和嵌入向量。接下来我们将观察并研究：**模型怎样逐 token 生成译文**。大致流程是：

```txt
英文原句 → 编码器 → 原文各位置的表示
已生成的中文前缀 → 解码器 → 下一个中文词元
```

编码器通过自注意力读取原文中的有效位置。解码器通过带因果掩码的自注意力读取目标前缀，再通过交叉注意力读取原文表示。每一步的解码器输出经词表投影得到各中文词元的预测分数，选取得分最大的一个 token，将它接到当前前缀末尾。


In [ ]:
def encode_sentence(model, sentence):
    tokens = tokenize_en(sentence)
    if not tokens or len(tokens) > 80:
        raise ValueError("请输入 1–80 个英文词元的短句。")
    ids = [BOS_ID] + ids_for(tokens, src_to_id) + [EOS_ID]
    device = next(model.parameters()).device
    return model.encode(torch.tensor([ids], device=device))


def next_token_log_probs(model, prefix_ids, memory, src_pad):
    prefix = torch.tensor([prefix_ids], device=memory.device)
    tgt_mask = library_causal_mask(prefix.size(1), prefix.device)
    logits = model.decode(prefix, memory, src_pad, tgt_mask)[0, -1].clone()
    logits[[PAD_ID, BOS_ID]] = -torch.inf
    return logits.log_softmax(dim=-1)


@torch.no_grad()
def beam_decode(model, sentence, beam_width=4, max_new_tokens=24):
    """按累计对数概率保留若干前缀，结束的候选不再扩展。"""
    if not 1 <= beam_width <= len(tgt_vocab) - 2:
        raise ValueError("束宽必须在 1 与有效候选词元数之间。")
    if not 0 <= max_new_tokens <= 80:
        raise ValueError("生成长度上限必须在 0–80 之间。")
    model.eval()
    memory, src_pad = encode_sentence(model, sentence)
    beams = [([BOS_ID], 0.0)]
    for _ in range(max_new_tokens):
        candidates = []
        for prefix, score in beams:
            if prefix[-1] == EOS_ID:
                candidates.append((prefix, score))
                continue
            log_probs = next_token_log_probs(model, prefix, memory, src_pad)
            scores, tokens = log_probs.topk(beam_width)
            candidates.extend(
                (prefix + [token], score + token_score)
                for token, token_score in zip(tokens.tolist(), scores.tolist())
            )
        beams = sorted(candidates, key=lambda item: item[1], reverse=True)[:beam_width]
        if all(prefix[-1] == EOS_ID for prefix, _ in beams):
            break
    return beams[0][0]


def as_translation(ids):
    return "".join(
        tgt_vocab[token] for token in ids if token not in {BOS_ID, EOS_ID, PAD_ID}
    )


def print_trace(ids):
    for step, token in enumerate(ids[1:], start=1):
        prefix = " ".join(tgt_vocab[i] for i in ids[:step])
        print(f"{step:2d}  {prefix}  →  {tgt_vocab[token]}")
    status = "生成 <eos>" if ids[-1] == EOS_ID else "达到长度上限"
    print(f"译文  {as_translation(ids)}（{status}）")

同学们可以修改下面的 `DEMO_SENTENCE`，观察不同的翻译结果。

> 由于这里模型和数据集都较小，所以只能处理一些简单的文本。当然，同学们也可以尝试一些长难文本，看看小模型在处理复杂文本的时候可能出现什么问题。


In [ ]:
# 可以修改这里的 DEMO_SENTENCE 观察不同输出
DEMO_SENTENCE = "I love you."
print("英文原句", DEMO_SENTENCE)
print_trace(beam_decode(reference_model, DEMO_SENTENCE, beam_width=1))

### 把参考译文错开一位

我们说注意力机制的优势在于，模型对单个词的学习，是基于词的上下文和关联信息来进行学习的，而与字词在文中的绝对位置无关。基于这个特性，我们提出下面这个问题：**如果我把译文词表错开一位，模型还能像之前那样进行学习吗？效果如何？**


In [ ]:
def make_target_batch(target_rows):
    # TODO 2 · 将一批目标序列在右侧补齐，再构造错开一位的输入和标签。
    # HINT: 这里使用现成函数 pad_sequence 进行填充
    # 每条序列已含 BOS、EOS。返回两个 LongTensor，形状均为 [批大小, 目标长度 - 1]。
    raise NotImplementedError

In [ ]:
check_target_rows = [
    torch.tensor([BOS_ID, 11, EOS_ID]),
    torch.tensor([BOS_ID, 21, 22, EOS_ID]),
]
check_inputs, check_labels = make_target_batch(check_target_rows)
torch.testing.assert_close(
    check_inputs, torch.tensor([[BOS_ID, 11, EOS_ID], [BOS_ID, 21, 22]])
)
torch.testing.assert_close(
    check_labels, torch.tensor([[11, EOS_ID, PAD_ID], [21, 22, EOS_ID]])
)
print("目标批次检查通过。标签保留 EOS，较短译文末尾多出的标签为 PAD。")

In [ ]:
def collate_batch(pairs):
    src_rows, tgt_rows = [], []
    for en, zh in pairs:
        src_rows.append(
            torch.tensor([BOS_ID] + ids_for(tokenize_en(en), src_to_id) + [EOS_ID])
        )
        tgt_rows.append(
            torch.tensor([BOS_ID] + ids_for(tokenize_zh(zh), tgt_to_id) + [EOS_ID])
        )
    src = pad_sequence(src_rows, batch_first=True, padding_value=PAD_ID)
    tgt_in, labels = make_target_batch(tgt_rows)
    return src, tgt_in, labels

In [ ]:
examples = [("I agree.", "我同意。"), ("I don't agree.", "我不同意。")]
src, tgt_in, labels = collate_batch(examples)

for i, (en, zh) in enumerate(examples):
    print(f"{en} → {zh}")
    print("英文词元  ", " ".join(src_vocab[index] for index in src[i].tolist()))
    print("解码器输入", " ".join(tgt_vocab[index] for index in tgt_in[i].tolist()))
    print("监督标签  ", " ".join(tgt_vocab[index] for index in labels[i].tolist()))

print("批次形状 [批大小, 序列长度]", src.shape, tgt_in.shape, labels.shape)
assert torch.equal(tgt_in[:, 1:], labels[:, :-1])

**记录 Q2**

- 推理轨迹里，选一步记录前缀和实际生成的词元。
- 训练「我同意。」时，预测「同」对应的输入位置和标签是什么？在这个位置能读到哪些中文词元？
- 两条译文长度不同时，哪一个标签位置应从损失中排除？说出你的判断依据。


## Q3 · 注意力权重和输出分别取决于什么？

**TODO 3**。实现 `scaled_dot_product_attention`。对给定的二维张量，计算每个查询对所有键的打分、归一化权重，以及值向量的加权和。再次复习 Attention 的公式：

$$
W = \operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right), \qquad O = WV .
$$

设 $Q$、$K$、$V$ 的形状分别为 $(L_q,d_k)$、$(L_k,d_k)$、$(L_k,d_v)$。按 `(output, weights)` 的顺序返回。想想 softmax 应该沿哪个维度执行，以及 $W$ 的第 $i$ 行对应哪个查询。

这里我们先实现一个注意力头。完整的翻译模型将调用 PyTorch 的多头注意力。在交叉注意力里，查询 $Q$ 来自解码器，键 $K$ 和值 $V$ 来自编码器输出。

用检查单元里的第一组例子，先手算一个查询的权重和加权输出。再做一次只替换 $V$ 的实验，预测权重和输出各自是否会改变。


In [ ]:
def scaled_dot_product_attention(q, k, v):
    """接收二维张量 q、k、v，返回（加权输出，注意力权重）。"""
    # TODO 3 · 计算缩放后的点积，沿键的位置做 softmax，再对 v 加权。
    raise NotImplementedError

In [ ]:
def check_attention():
    values = torch.tensor([[2.0, 4.0], [6.0, 8.0]])
    output, weights = scaled_dot_product_attention(
        torch.tensor([[1.0, 0.0]]), torch.zeros(2, 2), values
    )
    torch.testing.assert_close(weights, torch.tensor([[0.5, 0.5]]))
    torch.testing.assert_close(output, torch.tensor([[4.0, 6.0]]))

    # 查询的维度为 4，第一组点积为 [2, 0]，缩放后为 [1, 0]。
    output, weights = scaled_dot_product_attention(
        torch.tensor([[2.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]]),
        torch.tensor([[1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]]),
        values,
    )
    expected = torch.tensor([[0.7310586, 0.2689414], [0.5, 0.5]])
    torch.testing.assert_close(weights, expected)
    torch.testing.assert_close(output, expected @ values)
    print("注意力检查通过。每行权重对应一个查询，沿键的位置求和为 1。")
    print("注意力权重：", weights.tolist())


try:
    check_attention()
except NotImplementedError:
    print("请先补全 scaled_dot_product_attention，再运行本单元。")

In [ ]:
try:
    probe_q = torch.tensor([[1.0, 0.0]])
    probe_k = torch.zeros(2, 2)
    probe_v = torch.tensor([[2.0, 4.0], [6.0, 8.0]])
    changed_v = probe_v.clone()
    changed_v[1, 0] = 10.0
    original_output, original_weights = scaled_dot_product_attention(
        probe_q, probe_k, probe_v
    )
    changed_output, changed_weights = scaled_dot_product_attention(
        probe_q, probe_k, changed_v
    )
    print("权重", original_weights.tolist(), "→", changed_weights.tolist())
    print("输出", original_output.tolist(), "→", changed_output.tolist())
except NotImplementedError:
    print("先完成 Q3 的注意力函数，再运行这个对照。")

**记录 Q3**

写下手算结果、改变 $V$ 前后的权重和输出。用公式指出哪些量决定权重，哪些量决定被加权的信息。


## Q4 · 验证因果掩码

**TODO 4**。填写 `make_causal_mask(length, device)`，实现随机掩码。它返回布尔方阵，行是查询位置，列是键位置。在 `nn.Transformer` 中，`True` 表示屏蔽。

因果掩码屏蔽目标序列中比查询更靠后的位置。填充掩码屏蔽批次中补出来的 `<pad>` 键位置。模型将源端填充掩码同时传给编码器自注意力和解码器交叉注意力，并在解码器自注意力中使用目标端填充掩码。

检查时固定英文句子，在评估模式下只替换靠后的一个中文词元，比较更早位置的 logits。思考如果训练时允许读取这个被替换的位置，会发生什么。

跑完基础检查后，用同一个模型做一组有掩码与无掩码的对照。英文原句和目标前缀都固定，只替换一个后部词元。在运行前预测两组实验中，更早位置的 logits 最大绝对差会是多少或接近多少。


In [ ]:
def make_causal_mask(length, device):
    """返回长度为 length 的方形布尔掩码，True 表示禁止读取。"""
    # TODO 4 · 行是查询位置，列是键位置。当前位置和更早的位置都可以读取。
    raise NotImplementedError

In [ ]:
@torch.no_grad()
def check_causal_mask():
    expected = torch.tensor(
        [[False, True, True], [False, False, True], [False, False, False]],
        device=DEVICE,
    )
    torch.testing.assert_close(make_causal_mask(3, DEVICE), expected)

    src, tgt_in, _ = collate_batch([("I agree.", "我同意。")])
    src, tgt_in = src.to(DEVICE), tgt_in.to(DEVICE)
    changed = tgt_in.clone()
    changed_pos = tgt_in.size(1) - 1
    # 取词表中另一普通词元（同一条目标序列中的「我」）替换句号。
    changed[:, changed_pos] = tgt_in[:, 1]
    assert changed_pos > 1 and not torch.equal(changed, tgt_in), (
        "测试句需要不同的普通词元。"
    )

    reference_model.eval()
    mask = make_causal_mask(tgt_in.size(1), DEVICE)
    original_scores = reference_model(src, tgt_in, mask)
    changed_scores = reference_model(src, changed, mask)
    torch.testing.assert_close(
        original_scores[:, :changed_pos],
        changed_scores[:, :changed_pos],
        rtol=1e-5,
        atol=1e-5,
    )
    print("因果掩码检查通过。更改未来词元后，更早位置的预测分数保持一致。")
    print("目标端三阶掩码：", expected.to(torch.int).cpu().tolist())


try:
    check_causal_mask()
except NotImplementedError:
    print("请先补全 make_causal_mask，再运行本单元。")

In [ ]:
@torch.no_grad()
def measure_future_effect(english, chinese, change_pos, replacement):
    src_ids, target, _ = collate_batch([(english, chinese)])
    src_ids, target = src_ids.to(DEVICE), target.to(DEVICE)
    replacement_tokens = tokenize_zh(replacement)
    if len(replacement_tokens) != 1 or replacement_tokens[0] not in tgt_to_id:
        raise ValueError("请选一个目标词表里的中文词元作为替换值。")
    if not 1 <= change_pos < target.size(1):
        raise ValueError("请选择 BOS 之后、序列范围内的位置。位置从 0 开始。")
    replacement_id = tgt_to_id[replacement_tokens[0]]
    if replacement_id == target[0, change_pos].item():
        raise ValueError("替换词元需要和原词元不同。")

    changed = target.clone()
    changed[0, change_pos] = replacement_id
    reference_model.eval()
    mask = make_causal_mask(target.size(1), DEVICE)

    # 只比较替换位置之前的 logits。
    def earlier_difference(tgt_mask):
        before = reference_model(src_ids, target, tgt_mask)[:, :change_pos]
        after = reference_model(src_ids, changed, tgt_mask)[:, :change_pos]
        return (before - after).abs().max().item()

    return earlier_difference(mask), earlier_difference(None)


try:
    masked_gap, unmasked_gap = measure_future_effect("I agree.", "我同意。", 4, "我")
    print(
        f"早期 logits 的最大绝对差 | 有掩码 {masked_gap:.3e} | 无掩码 {unmasked_gap:.3e}"
    )
except NotImplementedError:
    print("先完成 Q4 的因果掩码，再运行这个对照。")

**记录 Q4**

记下预测和两组差值，解释为什么只看替换位置之前的 logits。从替换位置、替换词元或输入句对中选一项修改，再运行一次，说明你固定了什么、想排除哪一种解释。


## Q5 · 模型学到了什么？

**TODO 5**。实现 `token_loss`，把模型对每个位置的预测分数同标签对齐，计算非 `<pad>` 位置的交叉熵之和。先核对小张量上的手算结果。每轮统计时，整轮损失之和除以整轮有效词元数。

**TODO 6**。实现 `update_parameters`。对一个批次先清掉上一批的梯度，再对每个有效词元的平均损失反向传播。将全局梯度范数限制为 1.0 后执行优化器更新。先用一个参数验证更新量，再在同一个小批次上反复更新，最后训练翻译模型。

训练会调用你在 Q4 实现的掩码。默认跑 6 轮。可以先跑 1 轮、每轮 10 批，确认流程通了再用整份语料。


In [ ]:
def token_loss(logits, labels):
    # TODO 5 · 计算所有非 PAD 位置的词元交叉熵之和。
    # logits 形状为 [批大小, 目标长度, 词表大小]，labels 形状为 [批大小, 目标长度]。
    # 返回可求梯度的标量张量。
    raise NotImplementedError

In [ ]:
check_logits = torch.tensor(
    [[[0.0, 0.0, 0.0], [0.0, math.log(3.0), 0.0], [0.0, 6.0, 2.0]]],
    requires_grad=True,
)
check_targets = torch.tensor([[1, 2, PAD_ID]])
check_loss = token_loss(check_logits, check_targets)

# 前两个位置的正确类别概率分别为 1/3 和 1/5，最后一个位置是 PAD。
torch.testing.assert_close(check_loss, torch.tensor(math.log(3.0) + math.log(5.0)))
(gradient,) = torch.autograd.grad(check_loss, check_logits)
torch.testing.assert_close(gradient[0, -1], torch.zeros(3))
print("损失检查通过。有效词元的交叉熵已求和，PAD 位置的梯度为 0。")

In [ ]:
def update_parameters(model, optimizer, loss, valid_tokens):
    # TODO 6 · 完成一个训练批次的参数更新。
    # loss 是有效词元的损失之和，valid_tokens 是有效词元数，至少为 1。
    # 清梯度，对每词元平均损失反向传播，按全局梯度范数 1.0 裁剪，再更新参数。
    raise NotImplementedError

In [ ]:
def check_update_parameters():
    probe_model = nn.Linear(1, 1, bias=False)
    optimizer = torch.optim.SGD(probe_model.parameters(), lr=0.25)
    with torch.no_grad():
        probe_model.weight.fill_(2.0)
    # 模拟上一步残留的梯度。
    probe_model.weight.grad = torch.full_like(probe_model.weight, -20.0)

    # 12 / 4 = 3，裁剪到 1 后，权重由 2 变成 1.75。
    update_parameters(probe_model, optimizer, 12 * probe_model.weight.sum(), 4)
    torch.testing.assert_close(probe_model.weight, torch.tensor([[1.75]]))

    # 2 / 4 = 0.5，无需裁剪。清掉上一轮梯度后，权重变成 1.625。
    update_parameters(probe_model, optimizer, 2 * probe_model.weight.sum(), 4)
    torch.testing.assert_close(probe_model.weight, torch.tensor([[1.625]]))


check_update_parameters()
print("参数更新检查通过。清梯度、平均损失、梯度裁剪和参数更新顺序正确。")

In [ ]:
def make_loader(pairs, batch_size, shuffle=False):
    return DataLoader(
        pairs,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=torch.Generator().manual_seed(SEED),
        collate_fn=collate_batch,
    )


def run_epoch(model, loader, mask_fn, optimizer=None, max_batches=None):
    is_training = optimizer is not None
    model.train(is_training)
    loss_sum, token_count = 0.0, 0
    with torch.set_grad_enabled(is_training):
        for batch_number, batch in enumerate(loader, start=1):
            src, tgt_in, labels = (tensor.to(DEVICE) for tensor in batch)
            tgt_mask = mask_fn(tgt_in.size(1), tgt_in.device)
            logits = model(src, tgt_in, tgt_mask)
            loss = token_loss(logits, labels)
            valid_tokens = labels.ne(PAD_ID).sum().item()
            if is_training:
                update_parameters(model, optimizer, loss, valid_tokens)
            loss_sum += loss.detach().item()
            token_count += valid_tokens
            if max_batches is not None and batch_number >= max_batches:
                break
    return loss_sum / token_count

In [ ]:
# 调整为 1 轮、每轮 10 批，可以快速检查完整流程。
EPOCHS = 6
MAX_TRAIN_BATCHES = None  # None 表示使用整轮数据，或填写正整数限制每轮步数
BATCH_SIZE = 96

check_causal_mask()
train_mask = make_causal_mask

torch.manual_seed(SEED)
small_batch = train_pairs[:8]
small_loader = make_loader(small_batch, batch_size=len(small_batch))
small_model = new_model()
small_optimizer = torch.optim.AdamW(small_model.parameters(), lr=0.001)

batch_before = run_epoch(small_model, small_loader, train_mask)
for _ in range(15):
    run_epoch(small_model, small_loader, train_mask, small_optimizer)
batch_after = run_epoch(small_model, small_loader, train_mask)
print(f"同一小批次在评估模式下的损失 {batch_before:.3f} → {batch_after:.3f}")

In [ ]:
torch.manual_seed(SEED)
student_model = new_model()
optimizer = torch.optim.AdamW(student_model.parameters(), lr=0.001)
train_loader = make_loader(train_pairs, BATCH_SIZE, shuffle=True)
val_loader = make_loader(val_pairs, BATCH_SIZE)
history = []
start = time.perf_counter()

initial_val_loss = run_epoch(student_model, val_loader, train_mask)
print(f"初始验证损失 {initial_val_loss:.3f}")
for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(
        student_model, train_loader, train_mask, optimizer, MAX_TRAIN_BATCHES
    )
    val_loss = run_epoch(student_model, val_loader, train_mask)
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
    print(f"第 {epoch} 轮 | 训练 {train_loss:.3f} | 验证 {val_loss:.3f}")

print(f"累计用时 {time.perf_counter() - start:.1f} 秒")

**记录 Q5**

记下小批次更新前后的损失、初始验证损失，以及各轮训练与验证损失。训练前先写下预测。如果模型能记住一个小批次，但验证损失没有改善，这两项结果各说明什么？

验证损失衡量在参考译文前缀下预测下一个词元的表现。带着一个关于实际翻译的预测进入 Q6，再用生成结果核验。


## Q6 · 同一模型怎样生成不同译文？

**TODO 7**。推理时没有参考译文，译文前缀由模型已经生成的词元构成。填写 `greedy_decode`，从 `<bos>` 开始，反复选下一个概率最高的词元。生成 `<eos>` 或达到新词元数上限时结束，返回包含开头 `<bos>` 的 ID 列表。

骨架已将英文句子编码一次。`next_token_log_probs` 接收当前前缀和编码结果，返回词表中各词元的对数概率。用自己训练的权重输出一条完整的生成轨迹。

> 这里的 `greedy` 指的是**贪心**逻辑，也是一种经典的算法逻辑。其基本思想就是，对一个大问题，我在每一步都取到我当前步的最优解，直到全部解完，理论上我们就能拿到一个全局的，符合目标的最优解。


In [ ]:
@torch.no_grad()
def greedy_decode(model, sentence, max_new_tokens=24):
    """从 BOS 开始生成，返回含 BOS 和已生成词元的 ID 列表。"""
    model.eval()
    memory, src_pad = encode_sentence(model, sentence)
    prefix_ids = [BOS_ID]
    # TODO 7 · 用 next_token_log_probs(model, prefix_ids, memory, src_pad)
    # 取得下一词元的分数，选择最高分的 ID 并追加。生成 EOS 即结束。
    # max_new_tokens 仅计入新生成的词元。
    raise NotImplementedError

In [ ]:
def check_greedy_decode():
    limit = 24
    generated = greedy_decode(reference_model, DEMO_SENTENCE, max_new_tokens=limit)
    assert isinstance(generated, list) and all(
        isinstance(token, int) for token in generated
    )
    assert 2 <= len(generated) <= 1 + limit and generated[0] == BOS_ID
    assert all(token not in (BOS_ID, PAD_ID) for token in generated[1:])
    assert EOS_ID not in generated[:-1], "遇到 EOS 后应当停止生成。"
    if generated[-1] != EOS_ID:
        assert len(generated) == 1 + limit, "未生成 EOS 时应运行到长度上限。"

    one_step = greedy_decode(reference_model, DEMO_SENTENCE, max_new_tokens=1)
    assert len(one_step) == 2 and one_step == generated[:2], (
        "检查新生成词元的长度上限。"
    )
    width_one = beam_decode(
        reference_model, DEMO_SENTENCE, beam_width=1, max_new_tokens=limit
    )
    assert generated == width_one, "束宽为 1 时应与贪心解码一致。"
    print("贪心解码检查通过。输出符合长度与停止条件，且与束宽为 1 的结果一致。")


try:
    check_greedy_decode()
except NotImplementedError:
    print("请先补全 greedy_decode，再运行本单元。")

In [ ]:
# 可以替换成另一条想翻译的英文短句。
sentence_to_translate = DEMO_SENTENCE
try:
    print("英文原句", sentence_to_translate)
    print_trace(greedy_decode(student_model, sentence_to_translate))
except NotImplementedError:
    print("请先补全 greedy_decode，再观察自己训练的模型。")

### 固定模型，对比两种解码策略

下面三组成对测试句和一条额外对照句都从训练集和验证集中留出。中文参考译文由实验给出，表意一致的译法可以有多种。贪心每步只保留一个前缀，束搜索每步保留最多四个前缀，按累计对数概率排序。

两种策略使用同一组参考权重和相同的生成长度上限。分别检查否定、数量、主客体互换。对照输出，选一条句子分析译文丢失或改变了什么信息，并说明更换解码策略后这一处有什么变化。

先从否定、数量和人物关系中选一组，写出原文只改变了什么事实、这个事实应怎样出现在译文里。观察现成测试句后，再自拟一组只改变一个语义因素的短句。用下面的检查确认英文词元都在词表里，且原句未出现在训练集或验证集。


In [ ]:
def compare_decoding(model, cases):
    for category, en, zh in cases:
        unknown = [token for token in tokenize_en(en) if token not in src_to_id]
        print(f"[{category}] {en}  |  未登录英文词元：{unknown}")
        print("  参考译文", zh)
        print("  贪心解码", as_translation(greedy_decode(model, en, max_new_tokens=24)))
        print(
            "  束搜索  ",
            as_translation(beam_decode(model, en, beam_width=4, max_new_tokens=24)),
        )


try:
    compare_decoding(reference_model, [*TEST_CASES, DECODE_EXAMPLE])
except NotImplementedError:
    print("请先补全 greedy_decode，再进行两种解码策略的对照。")

In [ ]:
# 填写自己设计的两条英文短句和对应的参考译文。
first_en, first_zh = "", ""
second_en, second_zh = "", ""

if first_en and second_en:
    custom_cases = [("自拟 A", first_en, first_zh), ("自拟 B", second_en, second_zh)]
    custom_keys = [source_key(en) for _, en, _ in custom_cases]
    assert all(custom_keys) and custom_keys[0] != custom_keys[1], (
        "需要两条不同的英文原句。"
    )
    assert not set(custom_keys) & (train_keys | val_keys), (
        "请换一组未出现在训练或验证语料里的原句。"
    )
    unknown = sorted(
        {
            token
            for _, en, _ in custom_cases
            for token in tokenize_en(en)
            if token not in src_to_id
        }
    )
    assert not unknown, f"请替换这些未登录词元：{unknown}"
    assert first_zh.strip() and second_zh.strip(), "请给两条短句都填写参考译文。"
    try:
        compare_decoding(reference_model, custom_cases)
    except NotImplementedError:
        print("先完成贪心解码，再比较自拟短句。")
else:
    print("请填写自己设计的两条英文短句及参考译文。")

**记录 Q6**

- 保留自己训练的模型的一条生成轨迹。
- 记录自拟句对改变的事实，以及参考模型在贪心解码和束搜索下生成的四条译文。
- 指出一处具体差异，说明原句的事实是被保留、漏掉还是改错了。这两句的结果能支持多大范围的判断？


## Q7 · 对比 RNN、GRU 和 LSTM

让三种循环单元完成同一个英译中任务。编码器按顺序读取英文词元，将最后的状态交给解码器。解码器用它作为初始状态，每一步读取一个中文词元，再预测下一个词元。

| 循环单元 | 步与步之间传递什么              | 如何更新                                                                               |
| -------- | ------------------------------- | -------------------------------------------------------------------------------------- |
| 普通 RNN | 隐藏状态 $h_t$                  | 用当前输入和上一步隐藏状态计算新的 $h_t$。这里使用 tanh。                              |
| GRU      | 隐藏状态 $h_t$                  | 重置门调节旧状态参与候选状态计算的程度，更新门调节保留旧状态和采用候选状态的比例。     |
| LSTM     | 隐藏状态 $h_t$ 和记忆状态 $c_t$ | 遗忘门调节保留多少旧记忆，输入门调节写入多少新信息，输出门调节从记忆状态读出多少信息。 |

三个循环模型的解码器都从编码器末状态读取源句信息。回想前面 Transformer 的交叉注意力，解码器生成一个中文词元时，能读取英文句子的哪些表示？普通 RNN、GRU 和 LSTM 的状态都沿时间步传递，因此训练时每步的状态依赖上一步。单向解码器在第 $t$ 步只能看到当前及更早的输入。

三种模型使用相同的词表、数据划分、嵌入维度、隐藏维度、批次顺序、优化器设置和训练轮数。门控结构带来的参数量差异会显示在结果里。这批语料每条英文最多 10 个词元，比较结果对应这次短句任务。  
[PyTorch RNN](https://docs.pytorch.org/docs/stable/generated/torch.nn.RNN.html) · [GRU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html) · [LSTM](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html)


In [ ]:
from torch.nn.utils.rnn import pack_padded_sequence

RECURRENT_CELLS = {"RNN": nn.RNN, "GRU": nn.GRU, "LSTM": nn.LSTM}


class RecurrentTranslator(nn.Module):
    def __init__(self, cell_type, hidden_size):
        super().__init__()
        self.src_embedding = nn.Embedding(
            len(src_vocab), hidden_size, padding_idx=PAD_ID
        )
        self.tgt_embedding = nn.Embedding(
            len(tgt_vocab), hidden_size, padding_idx=PAD_ID
        )
        self.to_vocab = nn.Linear(hidden_size, len(tgt_vocab))
        recurrent = RECURRENT_CELLS[cell_type]
        self.encoder = recurrent(hidden_size, hidden_size, batch_first=True)
        self.decoder = recurrent(hidden_size, hidden_size, batch_first=True)

    def encode(self, src):
        lengths = src.ne(PAD_ID).sum(dim=1).cpu()
        # 跳过右侧填充，使末状态对应每句最后一个有效词元。
        packed = pack_padded_sequence(
            self.src_embedding(src), lengths, batch_first=True, enforce_sorted=False
        )
        _, state = self.encoder(packed)
        return state

    def decode(self, tgt_in, state):
        hidden, next_state = self.decoder(self.tgt_embedding(tgt_in), state)
        return self.to_vocab(hidden), next_state

    def forward(self, src, tgt_in, _tgt_mask):
        # 单向递推已经限定了可见前缀。保留掩码参数以复用前面的训练循环。
        logits, _ = self.decode(tgt_in, self.encode(src))
        return logits


@torch.no_grad()
def recurrent_greedy_decode(model, sentence, max_new_tokens=24):
    model.eval()
    device = next(model.parameters()).device
    src_ids = [BOS_ID, *ids_for(tokenize_en(sentence), src_to_id), EOS_ID]
    src = torch.tensor([src_ids], device=device)
    state = model.encode(src)
    output_ids = [BOS_ID]
    for _ in range(max_new_tokens):
        last_token = torch.tensor([[output_ids[-1]]], device=device)
        logits, state = model.decode(last_token, state)
        scores = logits[0, -1]
        scores[[PAD_ID, BOS_ID]] = -torch.inf
        next_id = scores.argmax().item()
        output_ids.append(next_id)
        if next_id == EOS_ID:
            break
    return output_ids

### 先看参数量，再比较结果

本题为必做题。运行下面两个单元。观察三个模型在同一个验证集上的词元平均交叉熵，再用贪心解码比较同一组测试句。循环层参数量包含编码器和解码器。


In [ ]:
RUN_RECURRENT_COMPARISON = True
RECURRENT_EPOCHS = 4
RECURRENT_BATCH_SIZE = 96
RECURRENT_HIDDEN_SIZE = 96
RECURRENT_MAX_BATCHES = None  # 填入正整数可以缩短每轮训练

recurrent_models = {}

if RUN_RECURRENT_COMPARISON:
    val_loader_recurrent = make_loader(val_pairs, RECURRENT_BATCH_SIZE)
    print("模型 | 循环层参数 | 总参数 | 初始验证损失 | 各轮验证损失")
    for cell_type in RECURRENT_CELLS:
        torch.manual_seed(SEED)
        model = RecurrentTranslator(cell_type, RECURRENT_HIDDEN_SIZE).to(DEVICE)
        train_loader_recurrent = make_loader(
            train_pairs, RECURRENT_BATCH_SIZE, shuffle=True
        )
        recurrent_optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
        initial_loss = run_epoch(
            model, val_loader_recurrent, lambda _length, _device: None
        )
        val_losses = []
        for _ in range(RECURRENT_EPOCHS):
            run_epoch(
                model,
                train_loader_recurrent,
                lambda _length, _device: None,
                optimizer=recurrent_optimizer,
                max_batches=RECURRENT_MAX_BATCHES,
            )
            val_losses.append(
                run_epoch(model, val_loader_recurrent, lambda _length, _device: None)
            )
        recurrent_params = sum(
            p.numel()
            for layer in (model.encoder, model.decoder)
            for p in layer.parameters()
        )
        all_params = sum(p.numel() for p in model.parameters())
        losses = " → ".join(f"{loss:.3f}" for loss in val_losses)
        print(
            f"{cell_type} | {recurrent_params:,} | {all_params:,} | {initial_loss:.3f} | {losses}"
        )
        recurrent_models[cell_type] = model
else:
    print("Q7 为必做题，请保持 RUN_RECURRENT_COMPARISON 为 True 并运行本单元。")

In [ ]:
if recurrent_models:
    for category, en, zh in TEST_CASES:
        print(f"[{category}] {en}  |  参考译文 {zh}")
        for cell_type, model in recurrent_models.items():
            ids = recurrent_greedy_decode(model, en, max_new_tokens=24)
            print(f"  {cell_type:4} {as_translation(ids)}")

**记录结果**

1. 三种模型的循环层各有多少参数？同样的隐藏维度下，为什么数量不同？
2. 选一组否定、数量或主客体互换的测试句，比较三个模型保留了哪些原文信息。验证损失的排序和这组译文的表现一致吗？
3. 任选一个循环模型和前面的 Transformer，画出生成首个中文词元时原文信息传到解码器的路径。


## 参考资料

- [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
- [PyTorch nn.Transformer](https://docs.pytorch.org/docs/stable/generated/torch.nn.Transformer.html)
- [ManyThings 汇编的 Tatoeba 英中平行语料](https://www.manythings.org/anki/)。课堂语料经过筛选、去重和划分，保留原文与逐句署名，中文包含简繁体。[CC BY 2.0 France](https://creativecommons.org/licenses/by/2.0/fr/)
